# Yaye POC — Projection SQL (MariaDB) → Knowledge Graph Neo4j

> **Banc d'essai exploratoire.** Valide le mapping **COMPLET** de la spec [`02-knowledge-graph-neo4j.md`](../../.agent_context/specs/yaye/02-knowledge-graph-neo4j.md) avant le pipeline de prod (Lot 1, TypeScript).

🔒 **Invariant** : `SQL → projection → Neo4j` uniquement. Rien ne naît dans le graphe. Tout est `MERGE` → rejouable.

## Pré-requis — **LOCAL d'abord** (cloud à la fin)
1. **Stack docker** (racine du repo) : `docker compose up -d mariadb neo4j`
2. **Dump chargé** dans la base dédiée `yaye_poc` (MariaDB projet `guichet_mariadb`, port hôte **3307**) :
   ```bash
   docker exec -i guichet_mariadb mariadb -uroot -proot \
     -e "CREATE DATABASE IF NOT EXISTS yaye_poc CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
         GRANT ALL ON yaye_poc.* TO 'guichet'@'%'; FLUSH PRIVILEGES;"
   sed 's/utf8mb4_0900_ai_ci/utf8mb4_unicode_ci/g' \
     /Users/macbookpro/Documents/GitHub/cjs/yaye-kg-poc/data/dump-railway-20260616T130722Z.sql \
     | docker exec -i guichet_mariadb mariadb -uroot -proot yaye_poc
   ```
3. **`.env`** : bloc **LOCAL actif** (Neo4j `bolt://localhost:7687`, MariaDB `yaye_poc@3307`). Le bloc **AURA reste commenté** → on bascule au cloud à l'étape J.

## Couverture — **aucun élément du périmètre écarté**
Projette **tout ce qui a une source** dans le dump (17 types de nœuds + opportunités décompressées en 10 sous-types ; relations REQUIERT/DEVELOPPE, RELEVE_DE/SITUE_A sur **toutes** les entités, MAITRISE/ATTESTE/PREPARE). Ce qui n'a **pas encore de source** (biblio physique → Lot 3, zone véhicule → Lot 2) est **listé explicitement** à l'étape I — jamais inventé (invariant read-model).

**Étapes** : 0 setup → A contraintes → B nœuds → C opportunités décompressées → D salles/véhicules → E enums réifiés + RELEVE_DE/SITUE_A → F relations FK → **F-bis DEVELOPPE** → G bénéficiaire + MAITRISE → **G-bis ATTESTE/PREPARE** → H validation → **I couverture du périmètre** → **J bascule cloud**.

## 0 — Setup : env, connexions, contrôle de la source

In [ ]:
import sys, os
from pathlib import Path

# rendre src/ importable, indépendamment du cwd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from src import mapping as M
from src import graph_loader as G

engine = G.mariadb_engine()
driver = G.neo4j_driver()
driver.verify_connectivity()
print('MariaDB tables:', len(G.list_tables(engine)))
print('Neo4j OK ->', os.environ['NEO4J_URI'])

In [ ]:
# Contrôle source : comptages des tables clés (sert de référence pour la validation finale)
SOURCE_COUNTS = {t: G.count_rows(engine, t) for t in [
    'opportunites','utilisateurs','profils_jeunes','skills','tags','programmes',
    'opportunite_types','evenements','ressources','centres','candidatures',
    'opportunites_skills','opportunites_tags','inscriptions_evenements'] if t in G.list_tables(engine)}
SOURCE_COUNTS

## A — Contraintes d'unicité (toujours en premier)

In [ ]:
labels_keys = {cfg['label']: cfg['key'] for cfg in M.NODES.values()}
labels_keys['Opportunite'] = 'id'
labels_keys['Salle'] = 'id'
labels_keys['Vehicule'] = 'id'
for r in M.REIFIED_ENUMS.values():
    labels_keys[r['label']] = r['key']
G.ensure_constraints(driver, labels_keys)
print('Contraintes posées:', list(labels_keys))

## B — Nœuds simples (1 table → 1 label)

In [ ]:
def project_simple(cfg):
    df = G.read_table(engine, cfg['table'], list(cfg['props'].keys()))
    df = df.rename(columns=cfg['props'])           # colonne SQL -> propriété graphe
    rows = df.to_dict('records')
    G.merge_nodes(driver, cfg['label'], cfg['key'], rows)
    return len(rows)

for name, cfg in M.NODES.items():
    n = project_simple(cfg)
    print(f"{cfg['label']:24} {n:>6}")

## C — Opportunités décompressées : `:Opportunite` + 1 label sous-type

Props communes sur `:Opportunite`, puis pour chaque sous-type on pose le label additionnel et les props CTI sur le **même** nœud (clé `id`).

In [ ]:
# 1) noeud commun
df_opp = G.read_table(engine, M.OPPORTUNITE_COMMON['table'], list(M.OPPORTUNITE_COMMON['props'].keys()))
df_opp = df_opp.rename(columns=M.OPPORTUNITE_COMMON['props'])
G.merge_nodes(driver, 'Opportunite', 'id', df_opp.to_dict('records'))
print('Opportunite (commun):', len(df_opp))

# 2) sous-types : SET label + props sur le meme noeud id
for label, (table, props) in M.OPPORTUNITE_SUBTYPES.items():
    if table not in G.list_tables(engine):
        print(f'  (skip {label}: table absente)'); continue
    cols = ['opportunite_id'] + list(props.keys())
    df = G.read_table(engine, table, cols).rename(columns={**props, 'opportunite_id': 'id'})
    # MERGE sur :Opportunite(id) existant + ajoute le label de sous-type
    G.merge_nodes(driver, 'Opportunite', 'id', df.to_dict('records'), extra_labels=[label])
    print(f'  :{label:14} {len(df):>5}')

## D — Ressources physiques de centre : `:Salle` / `:Vehicule`

In [ ]:
rc = M.RESSOURCE_CENTRE
df_rc = G.read_table(engine, rc['table'], list(rc['props'].keys()))
df_rc = df_rc.rename(columns=rc['props'])
for type_val, label in rc['labels'].items():
    sub = df_rc[df_rc['type'] == type_val]
    if len(sub):
        G.merge_nodes(driver, label, 'id', sub.to_dict('records'))
    print(f'  :{label:10} {len(sub):>4}')

## E — Enums réifiés en nœuds : `:Secteur` (domaine) & `:Region`

Pas de table source : on collecte les valeurs distinctes, on crée les nœuds, puis on rattache via `RELEVE_DE` (domaine) et `SITUE_A` (region).

In [ ]:
# Nœuds enums réifiés (valeurs distinctes) puis liens RELEVE_DE / SITUE_A — COMPLETS (Opp/Centre/Organisation)
def distinct_values(columns):
    vals = set()
    for table, col in columns:
        if table in G.list_tables(engine):
            df = G.read_table(engine, table, [col])
            vals |= {v for v in df[col].dropna().unique()}
    return sorted(vals)

for name, cfg in M.REIFIED_ENUMS.items():
    vals = distinct_values(cfg['from_columns'])
    G.merge_nodes(driver, cfg['label'], cfg['key'], [{cfg['key']: v} for v in vals])
    print(f"{cfg['label']:10} {len(vals):>3}  {vals}")

print('\nLiens vers enums réifiés:')
for lk in M.REIFIED_LINKS:
    if lk['table'] not in G.list_tables(engine):
        print(f"  (skip {lk['rel']} {lk['from_label']}: {lk['table']} absente)"); continue
    df = G.read_table(engine, lk['table'], [lk['id_col'], lk['val_col']]).dropna()
    pairs = [{'from': r[lk['id_col']], 'to': r[lk['val_col']]} for r in df.to_dict('records')]
    G.merge_rels(driver, lk['rel'], lk['from_label'], lk['id_col'], lk['to_label'], lk['to_key'], pairs)
    print(f"  {lk['rel']:10} {lk['from_label']:13} -> {lk['to_label']:8} {len(pairs):>6}")

## F — Relations issues de FK / jonctions

Note : `diplomes`/`experiences`/`certificats_moodle` référencent `profil_id` (→ `profils_jeunes.id`), mais le nœud `:Beneficiaire` est clé `cjs_uid`. On résout via une table de correspondance `profil_id → cjs_uid`.

In [ ]:
# Correspondance profil_id -> cjs_uid (pour les relations 'via_profil')
pj = G.read_table(engine, 'profils_jeunes', ['id','cjs_uid'])
PROFIL2UID = dict(zip(pj['id'], pj['cjs_uid']))

def project_relation(spec):
    table = spec['table']
    if table not in G.list_tables(engine):
        print(f"  (skip {spec['rel']}: {table} absente)"); return 0
    edge = spec.get('edge_props', {})
    cols = list({spec['fk_from'], spec['fk_to'], *edge.keys()})
    where = spec.get('where')
    where_sql = ' AND '.join(f"`{k}` = '{v}'" for k, v in where.items()) if where else None
    df = G.read_table(engine, table, cols, where_sql)
    pairs = []
    for r in df.to_dict('records'):
        frm = r[spec['fk_from']]
        if spec.get('via_profil'):
            frm = PROFIL2UID.get(frm)              # profil_id -> cjs_uid
        if frm is None or r[spec['fk_to']] is None:
            continue
        p = {'from': frm, 'to': r[spec['fk_to']]}
        p.update({prop: r[col] for col, prop in edge.items()})
        pairs.append(p)
    G.merge_rels(driver, spec['rel'], spec['from'][0], spec['from'][1],
                 spec['to'][0], spec['to'][1], pairs)
    return len(pairs)

for spec in M.RELATIONS:
    n = project_relation(spec)
    tag = ' (DISPOSE_DE/' + spec.get('where',{}).get('type','') + ')' if spec.get('where') else ''
    print(f"{spec['rel']:14}{tag:20} {n:>6}")

In [ ]:
# F-bis — DEVELOPPE : (:Formation) → Competence  (opportunites_skills.requise=0, restreint aux Formation)
ds = M.DEVELOPPE_SPEC
if ds['table'] in G.list_tables(engine) and 'opportunites_formation' in G.list_tables(engine):
    form_ids = set(G.read_table(engine, 'opportunites_formation', ['opportunite_id'])['opportunite_id'])
    where = ' AND '.join(f"`{k}` = '{v}'" for k, v in ds['where'].items())
    df = G.read_table(engine, ds['table'], [ds['fk_from'], ds['fk_to']], where)
    pairs = [{'from': r[ds['fk_from']], 'to': r[ds['fk_to']]}
             for r in df.to_dict('records')
             if r[ds['fk_from']] in form_ids and r[ds['fk_to']] is not None]
    G.merge_rels(driver, ds['rel'], ds['from'][0], ds['from'][1], ds['to'][0], ds['to'][1], pairs)
    print(f"DEVELOPPE (Formation→Competence): {len(pairs)} relations  "
          f"[{len(form_ids)} formations, opportunites_skills requise=0: {len(df)}]")
else:
    print('(skip DEVELOPPE: opportunites_skills / opportunites_formation absente)')

## G — Bénéficiaire enrichi + `MAITRISE` (dérivée du JSON `competences`)

`profils_jeunes.competences` est un JSON (`["Python","Agriculture"]`). On normalise en relations `MAITRISE` vers `:Competence` (matching sur libellé/slug). C'est la dérivation §3.B de la spec — version POC simple (matching exact insensible à la casse), à raffiner.

In [ ]:
# G-bis — ATTESTE (Certificat/Diplome→Competence) & PREPARE (Ressource→Competence)
# Dérivées par matching texte → :Competence. POC : containment insensible à la casse (à raffiner en flou).
skills_df = G.read_table(engine, 'skills', ['id', 'slug', 'libelle', 'categorie'])
COMP_IDX = [(str(f).strip().lower(), s.id)
            for s in skills_df.itertuples()
            for f in (s.libelle, s.slug, s.categorie) if f]

def match_competences(text):
    if not text:
        return []
    t = str(text).strip().lower()
    return list({sid for kw, sid in COMP_IDX if kw and (kw == t or kw in t or t in kw)})

for d in M.DERIVED_MATCH:
    if d['table'] not in G.list_tables(engine):
        print(f"  (skip {d['rel']} {d['from_label']}: {d['table']} absente)"); continue
    df = G.read_table(engine, d['table'], [d['id_col'], d['text_col']]).dropna(subset=[d['text_col']])
    pairs = [{'from': r[d['id_col']], 'to': sid}
             for r in df.to_dict('records')
             for sid in match_competences(r[d['text_col']])]
    G.merge_rels(driver, d['rel'], d['from_label'], d['id_col'], 'Competence', 'id', pairs)
    print(f"  {d['rel']:8} {d['from_label']:20} -> Competence : {len(pairs):>5} relations (sur {len(df)} lignes)")

In [ ]:
import json

# Enrichir :Beneficiaire avec niveauEtude / situationEmploi / completionScore
prof = G.read_table(engine, 'profils_jeunes',
                    ['cjs_uid','niveau_etude','situation_emploi','completion_score','competences'])
enrich = prof.rename(columns={'cjs_uid':'cjsUid','niveau_etude':'niveauEtude',
                              'situation_emploi':'situationEmploi','completion_score':'completionScore'})
G.merge_nodes(driver, 'Beneficiaire', 'cjsUid',
              enrich[['cjsUid','niveauEtude','situationEmploi','completionScore']].to_dict('records'))

# MAITRISE : competences (Json) -> :Competence (match libelle/slug, insensible casse)
skills = G.read_table(engine, 'skills', ['id','slug','libelle'])
by_label = {str(s.libelle).strip().lower(): s.id for s in skills.itertuples()}
by_slug  = {str(s.slug).strip().lower():    s.id for s in skills.itertuples()}

pairs, unmatched = [], set()
for r in prof.itertuples():
    raw = r.competences
    if not raw:
        continue
    try:
        comps = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        continue
    for c in (comps or []):
        k = str(c).strip().lower()
        sid = by_label.get(k) or by_slug.get(k)
        if sid:
            pairs.append({'from': r.cjs_uid, 'to': sid})
        else:
            unmatched.add(str(c))
G.merge_rels(driver, 'MAITRISE', 'Beneficiaire','cjsUid', 'Competence','id', pairs)
print(f'MAITRISE: {len(pairs)} relations | {len(unmatched)} libellés non appariés')
print('Exemples non appariés (à raffiner):', sorted(unmatched)[:15])

## H — Validation : comptages Neo4j vs SQL + requêtes de matching (spec §5)

In [ ]:
# 4) Reco collaborative (spec §5) — sortie AGRÉGÉE uniquement (jamais le cjsUid d'autrui)
if sample_ben:
    res = cypher('''
        MATCH (b:Beneficiaire {cjsUid:$uid})-[:A_POSTULE]->(:Opportunite)
              <-[:A_POSTULE]-(autre:Beneficiaire)-[:A_POSTULE]->(reco:Opportunite)
        WHERE NOT (b)-[:A_POSTULE]->(reco)
        RETURN reco.titre AS titre, count(*) AS popularite
        ORDER BY popularite DESC LIMIT 5
    ''', uid=sample_ben[0]['uid'])
    print('Reco collaborative:', res)

## Prochaines étapes

✅ **Couvert dans ce passage** : opportunités décompressées (10 sous-types) · Organisation complète · `REQUIERT` + `DEVELOPPE` · `RELEVE_DE`/`SITUE_A` sur **toutes** les entités · `MAITRISE` · `ATTESTE` · `PREPARE` · enums réifiés. L'étape **I** liste l'état de **chaque** nœud/relation du périmètre.

1. **J — Bascule cloud (Aura)** : commenter le bloc LOCAL / décommenter AURA dans `.env`, re-run le notebook (identique). C'est le seul changement.
2. **Raffiner les dérivées** `MAITRISE`/`ATTESTE`/`PREPARE` : le matching exact (containment) est grossier → passer au flou (embeddings/Levenshtein) une fois la pertinence mesurée. ⚠️ rappel dump : `opportunites_skills=0` et `profils_jeunes≈6` → ces relations sont **vides** (mécanique validée, pas la pertinence).
3. **Biblio physique** (`:Livre :ExemplaireLibre :Rayon :Emprunt` + `CONTIENT/A_EXEMPLAIRE/EST_LOCALISE_EN/SITUE_DANS/EMPRUNTE`) → **Lot 3** : créer d'abord les modèles Prisma (pas de source aujourd'hui).
4. **`ACCESSIBLE_A`** (zone véhicule) → **Lot 2** : ajouter la colonne `zoneRestriction` à `ressources_centre`.
5. **Porter en TypeScript** : ce mapping validé devient le pipeline de prod (Lot 1) — projection événementielle + sync nocturne, derrière `GraphPort`.

In [ ]:
# 3) Requête de la spec §5 — analyse d'écart de compétences pour une opportunité donnée
#    (remplacer $oppId / $uid par des valeurs réelles du dump)
sample_opp = cypher('MATCH (o:Opportunite)-[:REQUIERT]->(:Competence) RETURN o.id AS id LIMIT 1')
sample_ben = cypher('MATCH (b:Beneficiaire)-[:MAITRISE]->(:Competence) RETURN b.cjsUid AS uid LIMIT 1')
if sample_opp and sample_ben:
    res = cypher('''
        MATCH (o:Opportunite {id:$oppId})-[:REQUIERT {requise:true}]->(req:Competence)
        OPTIONAL MATCH (b:Beneficiaire {cjsUid:$uid})-[:MAITRISE]->(req)
        WITH req, b WHERE b IS NULL
        RETURN collect(req.libelle) AS competences_manquantes
    ''', oppId=sample_opp[0]['id'], uid=sample_ben[0]['uid'])
    print('Écart de compétences (exemple):', res)
else:
    print('Pas assez de données appariées pour l\'exemple — vérifier REQUIERT/MAITRISE.')

In [ ]:
# 4) Reco collaborative (spec §5) — sortie AGRÉGÉE uniquement (jamais le cjsUid d'autrui)
if sample_ben:
    res = cypher('''
        MATCH (b:Beneficiaire {cjsUid:$uid})-[:A_POSTULE]->(:Opportunite)
              <-[:A_POSTULE]-(autre:Beneficiaire)-[:A_POSTULE]->(reco:Opportunite)
        WHERE NOT (b)-[:A_POSTULE]->(reco)
        RETURN reco.titre AS titre, count(*) AS popularite
        ORDER BY popularite DESC LIMIT 5
    ''', uid=sample_ben[0]['uid'])
    print('Reco collaborative:', res)

driver.close()
print('\nTerminé. Graphe inspectable dans Neo4j Aura (Bloom / Browser).')

## Prochaines étapes (après validation de ce POC)

1. **Raffiner les dérivées** `MAITRISE` / `ATTESTE` / `PREPARE` (matching flou libellé↔compétence, enrichissement Moodle — spec §10.2).
2. **DEVELOPPE** vs **REQUIERT** : distinguer via `opportunites_skills.requise=false` sur les sous-types `:Formation`.
3. **Compléter Organisation** (`secteur`, `region`, `estVerifie`) une fois les colonnes confirmées sur le dump.
4. **Mesurer** : latence des requêtes de matching, qualité des recommandations sur données réelles.
5. **Porter en TypeScript** : ce mapping validé devient le pipeline de prod (Lot 1) — projection événementielle + sync nocturne, derrière `GraphPort` (cf. roadmap §Lot 1).